### ver2의 sample분석(MBPP)_prompt_v4
입력프롬프트에 힌트 추가   
ISSUE : LM디코딩 실패 패턴 발견 -> prompt를 좀 더 간결히 수정

In [1]:
import json
from collections import Counter
import statistics
import pandas as pd

# 경로를 여기서 직접 지정
path = "../../results/phase1_ver2/sample_mbpp/single_prompt_v4/details.jsonl"

def load_results(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = load_results(path)
len(data)

df = pd.DataFrame(data)
df.head(3)

,dataset,task_id,method,model_name,attempt_idx,prompt,raw_output,generated_code,status,passed,exec_success,error_type,error_message,latency_sec,meta,timeout
0,mbpp,601,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,You are a Python programmer.\nWrite a complete...,"```python\nclass Pair:\n def __init__(self,...","class Pair:\n def __init__(self, x, y):\n ...",pass,True,True,None,None,1.944421,"{'code_exec_passed': True, 'setup_exec_passed'...",False
1,mbpp,602,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,You are a Python programmer.\nWrite a complete...,"assert first_repeated_char(""abacddbec"") == ""b""...",def first_repeated_char(s):\n # Your code here,fail,False,False,indentation_error,"Traceback (most recent call last):\n File ""/h...",1.038059,"{'code_exec_passed': False, 'setup_exec_passed...",False
2,mbpp,603,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,You are a Python programmer.\nWrite a complete...,"assert get_ludic(15) == [1, 2, 3, 5, 7, 11, 13...",def get_ludic(n):\n # Your code here,fail,False,False,indentation_error,"Traceback (most recent call last):\n File ""/h...",1.115895,"{'code_exec_passed': False, 'setup_exec_passed...",False


In [2]:
# 전체 요약
total = len(data)
passed = sum(1 for x in data if x["status"] == "pass")
timeout = sum(1 for x in data if x["status"] == "timeout")
failed = total - passed - timeout

print("=" * 60)
print(f"📂 File: {path}")
print("📊 Overall")
print(f"Total: {total}")
print(f"Pass: {passed}")
print(f"Fail: {failed}")
print(f"Timeout: {timeout}")
print(f"Pass@1: {passed / total:.4f}")

📂 File: ../../results/phase1_ver2/sample_mbpp/single_prompt_v4/details.jsonl
📊 Overall
Total: 5
Pass: 3
Fail: 2
Timeout: 0
Pass@1: 0.6000


In [3]:
# error distribution
error_counter = Counter(
    x["error_type"] for x in data if x["error_type"] is not None
)

print("\n📉 Error Type Distribution")
for k, v in error_counter.most_common():
    print(f"{k}: {v}")
    
latencies = [x["latency_sec"] for x in data if x["latency_sec"] is not None]

if latencies:
    print("\n⏱ Latency")
    print(f"Avg: {statistics.mean(latencies):.3f}s")
    print(f"Max: {max(latencies):.3f}s")


📉 Error Type Distribution
indentation_error: 2

⏱ Latency
Avg: 1.272s
Max: 1.944s


### 실패한 샘플의 generated_code(모델이 생성한 str) 확인

In [7]:
failures = [x for x in data if x["status"] == "fail"][:2]

for f in failures:
    print("=" * 80)
    print(f"🧪 TASK: {f['task_id']}")
    print(f"🧪 ERROR: {f['error_type']}")
    print("\n============= RAW INPUT (head) =============")
    print(f["prompt"])
    print("\n============= RAW OUTPUT (head) =============")
    print(f["raw_output"])
    print("\n============= GENERATED CODE (head) =============")
    print(f["generated_code"])
    print("\n========================================================")

🧪 TASK: 602
🧪 ERROR: indentation_error

============= RAW INPUT (head) =============
You are a Python programmer.
Write a complete Python solution for the following problem.
Output only executable Python code.
Do not include any explanation, markdown fences, examples, or comments like 'Your code here'.
Your code must include all necessary definitions such as helper functions or classes.
Use the exact function name and argument signature required by the test.
Do not leave the function body empty.

Problem:
Write a python function to find the first repeated character in a given string.
Hint : 

You must write code that is compatible with the following test:
assert first_repeated_char("abcabc") == "a"


============= RAW OUTPUT (head) =============
assert first_repeated_char("abacddbec") == "b"
assert first_repeated_char("abcd") == None
```python
def first_repeated_char(s):
    # Your code here
```

















s


============= GENERATED CODE (head) =============
def first_repeated_c

현재 이슈는 Extraneous Token Error(placeholder,,) : 공백 문자나 무의미한 문자가 나열되는 상황   
LM 디코딩 실패 이슈   

In [ ]:
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     """
#     MBPP는 자연어 문제 설명만으로는 함수명/시그니처 mismatch가 자주 발생하므로,
#     테스트 힌트를 함께 제공해 인터페이스를 맞추도록 유도
#     """
#     visible_test_hint = ""
#     if sample.test_list:
#         visible_test_hint = (
#             "\nYou must write code that is compatible with the following test:\n"
#             f"{sample.test_list[0]}\n"
#         )

#     return (
#         "You are a Python programmer.\n"
#         "Write a complete Python solution for the following problem.\n"
#         "Output only executable Python code.\n"
#         "Do not include any explanation, markdown fences, examples, or comments like 'Your code here'.\n"
#         "Your code must include all necessary definitions such as helper functions or classes.\n"
#         "Use the exact function name and argument signature required by the test.\n"
#         "Do not leave the function body empty.\n\n"
#         f"Problem:\n{sample.problem_text}\n"
#         f"Hint : \n{visible_test_hint}"
#     )

In [ ]:
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     test_hint = sample.test_list[0] if sample.test_list else ""

#     return (
#         "Write Python code only.\n"
#         "Solve the problem below.\n"
#         "Use the exact function name and arguments required by the test.\n"
#         "Include any needed helper classes or functions.\n\n"
#         f"Problem:\n{sample.problem_text}\n\n"
#         f"Test hint:\n{test_hint}\n"
#     )